# 06 - DuckDB Backend

Nebula supports **DuckDB** as a first-class backend alongside Pandas, Polars, and Spark.

DuckDB relations are treated as **lazy** (like Spark), and Narwhals wraps them as `LazyFrame`, so most Narwhals-based transformers work automatically.

| Part | Topic |
|------|-------|
| **1** | Creating DuckDB Relations |
| **2** | Running Pipelines with DuckDB |
| **3** | Splits and Appending with DuckDB |
| **4** | Custom Transformers for DuckDB |
| **5** | Multi-Backend Comparison |

In [1]:
import duckdb
import narwhals as nw
import polars as pl

from nebula import TransformerPipeline
from nebula.transformers import *

---
## Part 1: Creating DuckDB Relations

DuckDB works with `DuckDBPyRelation` objects. You can create them from Python data, files, or SQL queries.

In [2]:
# Create a DuckDB connection (in-memory)
con = duckdb.connect()

# Create a relation from a SQL query
orders_duck = con.sql("""
    SELECT * FROM (VALUES
        (1, 'alice', 150.0, 'US', 'completed'),
        (2, 'bob', 75.0, 'EU', 'pending'),
        (3, 'alice', 200.0, 'US', 'completed'),
        (4, 'carol', 50.0, 'APAC', 'pending'),
        (5, 'bob', 300.0, 'EU', 'completed')
    ) AS t(order_id, customer, amount, region, status)
""")

print(type(orders_duck))
orders_duck

<class '_duckdb.DuckDBPyRelation'>


┌──────────┬──────────┬──────────────┬─────────┬───────────┐
│ order_id │ customer │    amount    │ region  │  status   │
│  int32   │ varchar  │ decimal(4,1) │ varchar │  varchar  │
├──────────┼──────────┼──────────────┼─────────┼───────────┤
│        1 │ alice    │        150.0 │ US      │ completed │
│        2 │ bob      │         75.0 │ EU      │ pending   │
│        3 │ alice    │        200.0 │ US      │ completed │
│        4 │ carol    │         50.0 │ APAC    │ pending   │
│        5 │ bob      │        300.0 │ EU      │ completed │
└──────────┴──────────┴──────────────┴─────────┴───────────┘

In [3]:
# You can also create from a Polars/Pandas DataFrame
orders_pl = pl.DataFrame({
    "order_id": [1, 2, 3, 4, 5],
    "customer": ["alice", "bob", "alice", "carol", "bob"],
    "amount": [150.0, 75.0, 200.0, 50.0, 300.0],
    "region": ["US", "EU", "US", "APAC", "EU"],
    "status": ["completed", "pending", "completed", "pending", "completed"],
})

orders_duck = duckdb.sql("SELECT * FROM orders_pl")
orders_duck

┌──────────┬──────────┬────────┬─────────┬───────────┐
│ order_id │ customer │ amount │ region  │  status   │
│  int64   │ varchar  │ double │ varchar │  varchar  │
├──────────┼──────────┼────────┼─────────┼───────────┤
│        1 │ alice    │  150.0 │ US      │ completed │
│        2 │ bob      │   75.0 │ EU      │ pending   │
│        3 │ alice    │  200.0 │ US      │ completed │
│        4 │ carol    │   50.0 │ APAC    │ pending   │
│        5 │ bob      │  300.0 │ EU      │ completed │
└──────────┴──────────┴────────┴─────────┴───────────┘

---
## Part 2: Running Pipelines with DuckDB

Nebula pipelines work with DuckDB relations just like with Polars or Pandas. Narwhals-based transformers (those implementing `_transform_nw`) work automatically.

In [4]:
pipe = TransformerPipeline(
    [
        RenameColumns(mapping={"customer": "customer_name"}),
        Filter(input_col="status", perform="keep", operator="eq", value="completed"),
        SelectColumns(columns=["order_id", "customer_name", "amount", "region"]),
    ],
    name="DuckDB Order Processing",
)

pipe.show(add_params=True)

*** DuckDB Order Processing *** (3 transformations)
 - RenameColumns -> PARAMS: mapping={'customer': 'customer_name'}
 - Filter -> PARAMS: input_col="status", operator="eq", perform="keep", value="completed"
 - SelectColumns -> PARAMS: columns=['order_id', 'customer_name', 'amount', 'region']


In [5]:
result = pipe.run(orders_duck)

print(f"Input type:  {type(orders_duck).__name__}")
print(f"Output type: {type(result).__name__}")
print()
result

2026-03-09 12:52:15,709 | [INFO]: Starting pipeline 'DuckDB Order Processing' 
2026-03-09 12:52:15,712 | [INFO]: Running 'RenameColumns' ... 
2026-03-09 12:52:15,719 | [INFO]: Completed 'RenameColumns' in 0.0s 
2026-03-09 12:52:15,724 | [INFO]: Running 'Filter' ... 
2026-03-09 12:52:15,727 | [INFO]: Completed 'Filter' in 0.0s 
2026-03-09 12:52:15,727 | [INFO]: Running 'SelectColumns' ... 
2026-03-09 12:52:15,728 | [INFO]: Completed 'SelectColumns' in 0.0s 
2026-03-09 12:52:15,729 | [INFO]: Pipeline 'DuckDB Order Processing' completed in 0.0s 


Input type:  DuckDBPyRelation
Output type: DuckDBPyRelation



┌──────────┬───────────────┬────────┬─────────┐
│ order_id │ customer_name │ amount │ region  │
│  int64   │    varchar    │ double │ varchar │
├──────────┼───────────────┼────────┼─────────┤
│        1 │ alice         │  150.0 │ US      │
│        3 │ alice         │  200.0 │ US      │
│        5 │ bob           │  300.0 │ EU      │
└──────────┴───────────────┴────────┴─────────┘

The output is a `DuckDBPyRelation` — Nebula preserves the backend type.

### 2.1 Using Functions with DuckDB

Python functions work too. Since DuckDB relations are wrapped in Narwhals as `LazyFrame`, use Narwhals or convert to native as needed.

In [6]:
def add_tax(df):
    """Works with any backend via Narwhals."""
    return nw.from_native(df).with_columns(
        (nw.col("amount") * 1.1).alias("amount_with_tax")
    )


pipe_with_func = TransformerPipeline(
    [
        add_tax,
        Filter(input_col="amount", perform="keep", operator="ge", value=100),
    ],
    name="DuckDB with Functions",
)

result = pipe_with_func.run(orders_duck)
result

2026-03-09 12:52:15,744 | [INFO]: Starting pipeline 'DuckDB with Functions' 
2026-03-09 12:52:15,745 | [INFO]: Running 'add_tax' ... 
2026-03-09 12:52:15,746 | [INFO]: Completed 'add_tax' in 0.0s 
2026-03-09 12:52:15,747 | [INFO]: Running 'Filter' ... 
2026-03-09 12:52:15,748 | [INFO]: Completed 'Filter' in 0.0s 
2026-03-09 12:52:15,748 | [INFO]: Pipeline 'DuckDB with Functions' completed in 0.0s 


┌──────────┬──────────┬────────┬─────────┬───────────┬────────────────────┐
│ order_id │ customer │ amount │ region  │  status   │  amount_with_tax   │
│  int64   │ varchar  │ double │ varchar │  varchar  │       double       │
├──────────┼──────────┼────────┼─────────┼───────────┼────────────────────┤
│        1 │ alice    │  150.0 │ US      │ completed │              165.0 │
│        3 │ alice    │  200.0 │ US      │ completed │ 220.00000000000003 │
│        5 │ bob      │  300.0 │ EU      │ completed │              330.0 │
└──────────┴──────────┴────────┴─────────┴───────────┴────────────────────┘

### 2.2 Nested Pipelines

Nested pipeline composition works identically with DuckDB.

In [7]:
cleaning = TransformerPipeline(
    [
        RenameColumns(mapping={"customer": "customer_name"}),
        DropNulls(),
    ],
    name="Cleaning",
)

enrichment = TransformerPipeline(
    [
        AddLiterals(data=[{"alias": "processed", "value": True}]),
        When(
            output_col="value_tier",
            conditions=[
                {"input_col": "amount", "operator": "ge", "value": 200, "output_constant": "high"},
                {"input_col": "amount", "operator": "ge", "value": 100, "output_constant": "medium"},
            ],
            otherwise_constant="low",
        ),
    ],
    name="Enrichment",
)

master = TransformerPipeline(
    [cleaning, enrichment],
    name="DuckDB Master Pipeline",
)

master.show(add_params=True)
result = master.run(orders_duck)
result

2026-03-09 12:52:15,769 | [INFO]: Starting pipeline 'DuckDB Master Pipeline' 
2026-03-09 12:52:15,775 | [INFO]: Running 'RenameColumns' ... 
2026-03-09 12:52:15,776 | [INFO]: Completed 'RenameColumns' in 0.0s 
2026-03-09 12:52:15,776 | [INFO]: Running 'DropNulls' ... 
2026-03-09 12:52:15,776 | [INFO]: Completed 'DropNulls' in 0.0s 
2026-03-09 12:52:15,776 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:52:15,776 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:52:15,776 | [INFO]: Running 'When' ... 
2026-03-09 12:52:15,776 | [INFO]: Completed 'When' in 0.0s 
2026-03-09 12:52:15,781 | [INFO]: Pipeline 'DuckDB Master Pipeline' completed in 0.0s 


*** DuckDB Master Pipeline *** (4 transformations)
*** Cleaning *** (2 transformations)
 - RenameColumns -> PARAMS: mapping={'customer': 'customer_name'}
 - DropNulls
*** Enrichment *** (2 transformations)
 - AddLiterals -> PARAMS: data=[{'alias': 'processed', 'value': True}]
 - When -> PARAMS: conditions=[{'input_col': 'amount', ' ... nstant="low", output_col="value_tier"


┌──────────┬───────────────┬────────┬─────────┬───────────┬───────────┬────────────┐
│ order_id │ customer_name │ amount │ region  │  status   │ processed │ value_tier │
│  int64   │    varchar    │ double │ varchar │  varchar  │  boolean  │  varchar   │
├──────────┼───────────────┼────────┼─────────┼───────────┼───────────┼────────────┤
│        1 │ alice         │  150.0 │ US      │ completed │ true      │ medium     │
│        2 │ bob           │   75.0 │ EU      │ pending   │ true      │ low        │
│        3 │ alice         │  200.0 │ US      │ completed │ true      │ high       │
│        4 │ carol         │   50.0 │ APAC    │ pending   │ true      │ low        │
│        5 │ bob           │  300.0 │ EU      │ completed │ true      │ high       │
└──────────┴───────────────┴────────┴─────────┴───────────┴───────────┴────────────┘

---
## Part 3: Splits and Appending with DuckDB

DuckDB uses its native `.union()` method for vertical concatenation instead of `pd.concat` or `pl.concat`. This is handled automatically by Nebula.

### 3.1 Split Pipeline

In [8]:
def regional_split(df):
    return df.filter(nw.col("region") == nw.lit("US")), "US"


pipe_split = TransformerPipeline(
    [
        AddLiterals(data=[{"alias": "tax_rate", "value": 0.08}]),
    ],
    split_function=regional_split,
    name="US Tax Pipeline",
)

result = pipe_split.run(orders_duck)
result

2026-03-09 12:52:15,800 | [INFO]: Starting pipeline 'US Tax Pipeline' 
2026-03-09 12:52:15,801 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:52:15,802 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:52:15,802 | [INFO]: Pipeline 'US Tax Pipeline' completed in 0.0s 


┌──────────┬──────────┬────────┬─────────┬───────────┬──────────┐
│ order_id │ customer │ amount │ region  │  status   │ tax_rate │
│  int64   │ varchar  │ double │ varchar │  varchar  │  double  │
├──────────┼──────────┼────────┼─────────┼───────────┼──────────┤
│        1 │ alice    │  150.0 │ US      │ completed │     0.08 │
│        2 │ bob      │   75.0 │ EU      │ pending   │     0.08 │
│        3 │ alice    │  200.0 │ US      │ completed │     0.08 │
│        4 │ carol    │   50.0 │ APAC    │ pending   │     0.08 │
│        5 │ bob      │  300.0 │ EU      │ completed │     0.08 │
└──────────┴──────────┴────────┴─────────┴───────────┴──────────┘

### 3.2 Appending with Missing Columns

DuckDB handles `allow_missing_columns=True` by inserting typed NULL columns via SQL expressions.

In [9]:
from nebula.storage import nebula_storage as ns

# Two DuckDB relations with different columns
df1 = duckdb.sql("SELECT 1 AS id, 'a' AS name")
df2 = duckdb.sql("SELECT 2 AS id, 100.0 AS amount")

ns.set("extra_data", df2)

pipe_append = TransformerPipeline(
    [
        AddLiterals(data=[{"alias": "source", "value": "branch"}]),
    ],
    branch={
        "storage": "extra_data",
        "end": "append",
    },
    allow_missing_columns=True,
    name="Append with Missing Columns",
)

result = pipe_append.run(df1)
result

2026-03-09 12:52:15,820 | [INFO]: Nebula Storage: setting an object (<class '_duckdb.DuckDBPyRelation'>) with the key "extra_data". 
2026-03-09 12:52:15,825 | [INFO]: Starting pipeline 'Append with Missing Columns' 
2026-03-09 12:52:15,825 | [INFO]: Entering branch 
2026-03-09 12:52:15,825 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:52:15,827 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:52:15,827 | [INFO]: Pipeline 'Append with Missing Columns' completed in 0.0s 


┌──────────────┬───────┬─────────┬─────────┐
│    amount    │  id   │  name   │ source  │
│ decimal(4,1) │ int32 │ varchar │ varchar │
├──────────────┼───────┼─────────┼─────────┤
│        100.0 │     2 │ NULL    │ branch  │
│         NULL │     1 │ a       │ NULL    │
└──────────────┴───────┴─────────┴─────────┘

---
## Part 4: Custom Transformers for DuckDB

### 4.1 Narwhals-Based (Recommended)

Most transformers should use `_transform_nw()` — this works with DuckDB automatically since Narwhals wraps DuckDB relations as `LazyFrame`.

In [10]:
from nebula.base import Transformer


class DoubleAmount(Transformer):
    """Works with all backends including DuckDB."""

    def _transform_nw(self, df):
        return df.with_columns(
            (nw.col("amount") * 2).alias("double_amount")
        )


t = DoubleAmount()
result = t.transform(orders_duck)
result

┌──────────┬──────────┬────────┬─────────┬───────────┬───────────────┐
│ order_id │ customer │ amount │ region  │  status   │ double_amount │
│  int64   │ varchar  │ double │ varchar │  varchar  │    double     │
├──────────┼──────────┼────────┼─────────┼───────────┼───────────────┤
│        1 │ alice    │  150.0 │ US      │ completed │         300.0 │
│        2 │ bob      │   75.0 │ EU      │ pending   │         150.0 │
│        3 │ alice    │  200.0 │ US      │ completed │         400.0 │
│        4 │ carol    │   50.0 │ APAC    │ pending   │         100.0 │
│        5 │ bob      │  300.0 │ EU      │ completed │         600.0 │
└──────────┴──────────┴────────┴─────────┴───────────┴───────────────┘

### 4.2 DuckDB-Specific Transformer

When you need DuckDB-specific features (e.g., raw SQL expressions), implement `_transform_duckdb()`:

In [11]:
class DuckDBSqlTransform(Transformer):
    """Use raw DuckDB SQL for transformation."""

    def __init__(self, *, sql_expr: str, alias: str):
        super().__init__()
        self._sql_expr = sql_expr
        self._alias = alias

    def _transform_duckdb(self, df):
        return df.select(f'*, ({self._sql_expr}) AS "{self._alias}"')


t = DuckDBSqlTransform(sql_expr="amount * 0.1", alias="tax")
result = t.transform(orders_duck)
result

┌──────────┬──────────┬────────┬─────────┬───────────┬────────┐
│ order_id │ customer │ amount │ region  │  status   │  tax   │
│  int64   │ varchar  │ double │ varchar │  varchar  │ double │
├──────────┼──────────┼────────┼─────────┼───────────┼────────┤
│        1 │ alice    │  150.0 │ US      │ completed │   15.0 │
│        2 │ bob      │   75.0 │ EU      │ pending   │    7.5 │
│        3 │ alice    │  200.0 │ US      │ completed │   20.0 │
│        4 │ carol    │   50.0 │ APAC    │ pending   │    5.0 │
│        5 │ bob      │  300.0 │ EU      │ completed │   30.0 │
└──────────┴──────────┴────────┴─────────┴───────────┴────────┘

---
## Part 5: Multi-Backend Comparison

The same pipeline works across all backends. The output type matches the input type.

In [12]:
pipe = TransformerPipeline(
    [
        Filter(input_col="status", perform="keep", operator="eq", value="completed"),
        AddLiterals(data=[{"alias": "processed", "value": True}]),
        AssertNotEmpty(df_name="Completed Orders"),
    ],
    name="Multi-Backend Pipeline",
)

# Run with Polars
result_pl = pipe.run(orders_pl)
print(f"Polars input  -> {type(result_pl).__name__} output")

# Run with Pandas
orders_pd = orders_pl.to_pandas()
result_pd = pipe.run(orders_pd)
print(f"Pandas input  -> {type(result_pd).__name__} output")

# Run with DuckDB
result_dk = pipe.run(orders_duck)
print(f"DuckDB input  -> {type(result_dk).__name__} output")

2026-03-09 12:52:15,886 | [INFO]: Starting pipeline 'Multi-Backend Pipeline' 
2026-03-09 12:52:15,886 | [INFO]: Running 'Filter' ... 
2026-03-09 12:52:15,897 | [INFO]: Completed 'Filter' in 0.0s 
2026-03-09 12:52:15,897 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:52:15,900 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:52:15,900 | [INFO]: Running 'AssertNotEmpty' ... 
2026-03-09 12:52:15,900 | [INFO]: Completed 'AssertNotEmpty' in 0.0s 
2026-03-09 12:52:15,900 | [INFO]: Pipeline 'Multi-Backend Pipeline' completed in 0.0s 
2026-03-09 12:52:15,900 | [INFO]: Starting pipeline 'Multi-Backend Pipeline' 
2026-03-09 12:52:15,905 | [INFO]: Running 'Filter' ... 
2026-03-09 12:52:15,916 | [INFO]: Completed 'Filter' in 0.0s 


Polars input  -> DataFrame output


2026-03-09 12:52:15,916 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:52:15,916 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:52:15,916 | [INFO]: Running 'AssertNotEmpty' ... 
2026-03-09 12:52:15,916 | [INFO]: Completed 'AssertNotEmpty' in 0.0s 
2026-03-09 12:52:15,916 | [INFO]: Pipeline 'Multi-Backend Pipeline' completed in 0.0s 
2026-03-09 12:52:15,916 | [INFO]: Starting pipeline 'Multi-Backend Pipeline' 
2026-03-09 12:52:15,921 | [INFO]: Running 'Filter' ... 
2026-03-09 12:52:15,921 | [INFO]: Completed 'Filter' in 0.0s 
2026-03-09 12:52:15,921 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:52:15,921 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:52:15,921 | [INFO]: Running 'AssertNotEmpty' ... 
2026-03-09 12:52:15,927 | [INFO]: Completed 'AssertNotEmpty' in 0.0s 
2026-03-09 12:52:15,927 | [INFO]: Pipeline 'Multi-Backend Pipeline' completed in 0.0s 


Pandas input  -> DataFrame output
DuckDB input  -> DuckDBPyRelation output


In [13]:
# All three produce the same data
print("Polars:")
print(result_pl)
print("\nDuckDB:")
result_dk

Polars:
shape: (3, 6)
┌──────────┬──────────┬────────┬────────┬───────────┬───────────┐
│ order_id ┆ customer ┆ amount ┆ region ┆ status    ┆ processed │
│ ---      ┆ ---      ┆ ---    ┆ ---    ┆ ---       ┆ ---       │
│ i64      ┆ str      ┆ f64    ┆ str    ┆ str       ┆ bool      │
╞══════════╪══════════╪════════╪════════╪═══════════╪═══════════╡
│ 1        ┆ alice    ┆ 150.0  ┆ US     ┆ completed ┆ true      │
│ 3        ┆ alice    ┆ 200.0  ┆ US     ┆ completed ┆ true      │
│ 5        ┆ bob      ┆ 300.0  ┆ EU     ┆ completed ┆ true      │
└──────────┴──────────┴────────┴────────┴───────────┴───────────┘

DuckDB:


┌──────────┬──────────┬────────┬─────────┬───────────┬───────────┐
│ order_id │ customer │ amount │ region  │  status   │ processed │
│  int64   │ varchar  │ double │ varchar │  varchar  │  boolean  │
├──────────┼──────────┼────────┼─────────┼───────────┼───────────┤
│        1 │ alice    │  150.0 │ US      │ completed │ true      │
│        3 │ alice    │  200.0 │ US      │ completed │ true      │
│        5 │ bob      │  300.0 │ EU      │ completed │ true      │
└──────────┴──────────┴────────┴─────────┴───────────┴───────────┘

---
## Summary

| Feature | DuckDB Support |
|---------|---------------|
| Flat pipelines | Yes |
| Split pipelines | Yes (uses `.union()` for merging) |
| Branching | Yes |
| Apply-to-rows | Yes |
| Narwhals transformers (`_transform_nw`) | Yes (wrapped as `LazyFrame`) |
| Backend-specific (`_transform_duckdb`) | Yes |
| `allow_missing_columns` | Yes (inserts typed NULL via SQL) |
| Config-driven pipelines | Yes |

**Key differences from other backends:**
- DuckDB relations are **lazy** (like Spark)
- Appending uses DuckDB's `.union()` instead of `concat`
- Missing columns are filled with `NULL::{type}` SQL expressions
- Narwhals wraps DuckDB relations as `LazyFrame`